# Agent Valuation

This notebook builds a deterministic valuation-agent workflow: retrieve evidence, run a valuation tool, compare with market price, and write a conclusion.

Abbreviations used in this notebook:

- **AI**: Artificial Intelligence.
- **DCF**: Discounted Cash Flow.
- **FCF**: Free Cash Flow.
- **WACC**: Weighted Average Cost of Capital.
- **RAG**: Retrieval-Augmented Generation.
- **EV**: Enterprise Value, the value of the operating business.
- **LLM**: Large Language Model.
- **TV**: Terminal Value.

## 1. Intuition

An analyst agent should not be just a chat interface. It needs tools, evidence, and a workflow. For valuation, the agent should retrieve relevant documents, calculate value, inspect assumptions, and produce a cautious conclusion.

## 2. Mathematics

Agent workflow:

$$
Question \rightarrow Retrieve \ Evidence \rightarrow Run \ Valuation \ Tool \rightarrow Compare \ Price \rightarrow Conclude
$$

DCF value per share:

$$
Value/Share = \frac{EV - NetDebt}{Shares}
$$

Upside or downside:

$$
Upside = \frac{Intrinsic\ Value}{Current\ Price} - 1
$$

Where:
- `Question` = user's valuation question.
- `Evidence` = retrieved financial context.
- `EV` = enterprise value from the valuation model.
- `NetDebt` = debt minus cash and cash equivalents.
- `Shares` = shares outstanding.
- `Intrinsic Value` = model-estimated value per share.
- `Current Price` = observed market price per share.


## 3. Implementation

We run a deterministic retrieve-value-conclude workflow. In production, each step could be tool-called by an LLM, but here each step is inspectable Python.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "06_ai_agents" / "ai_utils.py"
spec = importlib.util.spec_from_file_location("ai_utils", helper_path)
ai_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ai_utils)

plt.style.use("seaborn-v0_8-whitegrid")
documents = ai_utils.sample_financial_documents()

question = "Is NESN.SW undervalued based on cash flow quality, risks, and DCF assumptions?"
state = ai_utils.answer_valuation_question(question, current_price=96.0)
state.retrieved_documents[["doc_id", "section", "score", "text"]]

In [ ]:
valuation = state.valuation
valuation.to_frame("value").round(2)

In [ ]:
current_price = 96.0
upside = valuation["value_per_share"] / current_price - 1
print(f"Question: {state.question}")
print(f"Value per share: {valuation['value_per_share']:.2f}")
print(f"Current price: {current_price:.2f}")
print(f"Upside/downside: {upside:.1%}")
print(f"Conclusion: {state.conclusion}")

## 4. Visualization

The agent should expose what drove the answer: valuation components and retrieved evidence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
valuation[["pv_fcf", "pv_terminal"]].plot(kind="bar", ax=axes[0], color="#2f6f8f")
axes[0].set_title("DCF Value Components")
axes[0].set_ylabel("CHF millions")
axes[0].tick_params(axis="x", rotation=0)

state.retrieved_documents.sort_values("score").plot(x="doc_id", y="score", kind="barh", ax=axes[1], color="#9a6b2f", legend=False)
axes[1].set_title("Evidence Retrieval Scores")
axes[1].set_xlabel("Score")
plt.tight_layout(); plt.show()

## 5. Application

A real valuation agent should log assumptions, citations, tool outputs, and uncertainty. It should also refuse to present a single valuation as certainty.

In [ ]:
audit_log = pd.DataFrame([
    {"step": "retrieve", "output": f"{len(state.retrieved_documents)} evidence documents"},
    {"step": "value", "output": f"DCF value/share {valuation['value_per_share']:.2f}"},
    {"step": "compare", "output": f"Upside/downside {upside:.1%}"},
    {"step": "conclude", "output": state.conclusion},
])
audit_log

## 6. Reflection

- Agents need tools and evidence, not only language generation.
- Valuation conclusions should preserve uncertainty.
- Audit logs make agent reasoning reviewable.
- Retrieval quality can change the final answer.

Questions to answer after running the notebook:

1. Which evidence affected the conclusion most?
2. What valuation assumption would you stress test first?
3. What should the agent cite in a final report?
4. What guardrails should prevent overconfident recommendations?